# EDGAR Company Explorer
Use this notebook to browse available companies and decide which ones to include in the project.

**Goal:** Find companies with consistent 10-K filings from 2010 to present.

In [12]:
import requests
import pandas as pd
import time
from io import StringIO

HEADERS = {"User-Agent": "daphne s_hsueh25@stud.hwr-berlin.de"}
REQUEST_DELAY = 0.15

## Step 1 — Download the full EDGAR company list

This single endpoint returns every company that has ever filed with the SEC (~13,000 entries).
It includes: CIK number, ticker symbol, and company name.

In [13]:
def get_all_edgar_companies() -> pd.DataFrame:
    """
    Download the master company list from EDGAR.
    Returns DataFrame with columns: cik, ticker, company
    """
    url = "https://www.sec.gov/files/company_tickers.json"
    response = requests.get(url, headers=HEADERS)
    response.raise_for_status()
    data = response.json()

    df = pd.DataFrame.from_dict(data, orient="index")
    df.columns = ["cik", "ticker", "company"]
    df["cik"] = df["cik"].astype(str).str.zfill(10)  # zero-pad to 10 digits
    df = df.reset_index(drop=True)
    return df


all_companies = get_all_edgar_companies()

print(f"Total companies in EDGAR: {len(all_companies):,}")
all_companies.head(10)

Total companies in EDGAR: 10,400


,cik,ticker,company
0,0001045810,NVDA,NVIDIA CORP
1,0000320193,AAPL,Apple Inc.
2,0001652044,GOOGL,Alphabet Inc.
3,0000789019,MSFT,MICROSOFT CORP
4,0001018724,AMZN,AMAZON COM INC
5,0001730168,AVGO,Broadcom Inc.
6,0001326801,META,"Meta Platforms, Inc."
7,0001318605,TSLA,"Tesla, Inc."
8,0000723125,MU,MICRON TECHNOLOGY INC
9,0001067983,BRK-B,BERKSHIRE HATHAWAY INC


## Step 2 — Get S&P 500 list and cross-reference with EDGAR

The EDGAR list includes thousands of small filers. We narrow to S&P 500
because these are large, well-known companies with long, consistent filing histories.

In [14]:
def get_sp500() -> pd.DataFrame:
    """Fetch S&P 500 list from Wikipedia with browser-like headers to avoid 403."""
    url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
    wiki_headers = {"User-Agent": "Mozilla/5.0 (compatible; research project)"}
    response = requests.get(url, headers=wiki_headers)
    response.raise_for_status()

    df = pd.read_html(StringIO(response.text))[0]
    df = df[["Symbol", "Security", "GICS Sector", "CIK"]].copy()
    df.columns = ["ticker", "company", "sector", "cik_wiki"]
    df["ticker"] = df["ticker"].str.replace(".", "-", regex=False)
    df["cik_wiki"] = df["cik_wiki"].astype(str).str.zfill(10)
    return df


sp500 = get_sp500()

print(f"S&P 500 companies: {len(sp500)}")
print(f"\nSectors:")
print(sp500["sector"].value_counts().to_string())

S&P 500 companies: 503

Sectors:
sector
Industrials               80
Financials                76
Information Technology    72
Health Care               59
Consumer Discretionary    48
Consumer Staples          36
Utilities                 31
Real Estate               31
Materials                 26
Communication Services    23
Energy                    21


## Step 3 — Check filing counts per company (2010–present)

For each company, we query EDGAR's Submissions API to count how many
10-K filings exist from 2010 onwards. This tells us data availability.

**Start with a small sample** (20 companies) to get a feel for the data.
Set `SAMPLE_SIZE = None` to run on all S&P 500.

In [15]:
SAMPLE_SIZE = 0   # set to None for full S&P 500
START_YEAR  = 2010

sample = sp500.head(SAMPLE_SIZE) if SAMPLE_SIZE else sp500


def count_10k_filings(cik: str, start_year: int) -> dict:
    """
    Query EDGAR submissions API for a company and count 10-K filings
    from start_year to present.

    Returns dict with: total_10k, earliest_year, latest_year
    """
    url = f"https://data.sec.gov/submissions/CIK{cik}.json"
    try:
        response = requests.get(url, headers=HEADERS, timeout=10)
        response.raise_for_status()
        data = response.json()
        time.sleep(REQUEST_DELAY)
    except Exception as e:
        return {"total_10k": None, "earliest_year": None, "latest_year": None, "error": str(e)}

    filings = data.get("filings", {}).get("recent", {})
    if not filings or not filings.get("form"):
        return {"total_10k": 0, "earliest_year": None, "latest_year": None, "error": None}

    forms = pd.Series(filings["form"])
    dates = pd.to_datetime(pd.Series(filings["filingDate"]))

    # Filter to 10-K only, from start_year
    mask = (forms == "10-K") & (dates.dt.year >= start_year)
    filtered_dates = dates[mask]

    if filtered_dates.empty:
        return {"total_10k": 0, "earliest_year": None, "latest_year": None, "error": None}

    return {
        "total_10k":    int(mask.sum()),
        "earliest_year": int(filtered_dates.dt.year.min()),
        "latest_year":   int(filtered_dates.dt.year.max()),
        "error":         None,
    }


# Run the check
results = []
for _, row in sample.iterrows():
    counts = count_10k_filings(row["cik_wiki"], START_YEAR)
    results.append({
        "ticker":       row["ticker"],
        "company":      row["company"],
        "sector":       row["sector"],
        "cik":          row["cik_wiki"],
        **counts
    })

availability = pd.DataFrame(results)

print(f"Checked {len(availability)} companies")
availability.sort_values("total_10k", ascending=False)

Checked 503 companies


,ticker,company,sector,cik,total_10k,earliest_year,latest_year,error
457,ULTA,Ulta Beauty,Consumer Discretionary,0001403568,17,2010.0,2026.0,None
344,NVR,"NVR, Inc.",Consumer Discretionary,0000906163,17,2010.0,2026.0,None
195,FRT,Federal Realty Investment Trust,Real Estate,0000034903,17,2010.0,2026.0,None
284,LVS,Las Vegas Sands,Consumer Discretionary,0001300514,17,2010.0,2026.0,None
75,BRO,Brown & Brown,Financials,0000079282,16,2011.0,2026.0,None
...,...,...,...,...,...,...,...,...
358,PSKY,Paramount Skydance Corporation,Communication Services,0002041610,1,2026.0,2026.0,None
385,Q,Qnity Electronics,Information Technology,0002058873,1,2026.0,2026.0,None
110,C,Citigroup,Financials,0000831001,1,2026.0,2026.0,None
404,SNDK,Sandisk,Information Technology,0002023554,1,2025.0,2025.0,None


## Step 4 — Filter: keep only companies with full coverage

We keep companies that have filed consistently from 2010 to present.
"Full coverage" = at least 12 filings (roughly one per year since 2010).

In [16]:
MIN_FILINGS = 12  # adjust based on what you see above

well_covered = availability[
    (availability["total_10k"] >= MIN_FILINGS) &
    (availability["earliest_year"] <= 2011) &  # started filing by 2011
    (availability["error"].isna())
].copy()

print(f"Companies with full coverage (≥{MIN_FILINGS} filings from 2010): {len(well_covered)}")
print(f"\nBy sector:")
print(well_covered["sector"].value_counts().to_string())

well_covered

Companies with full coverage (≥12 filings from 2010): 14

By sector:
sector
Industrials               3
Consumer Discretionary    3
Health Care               2
Financials                2
Real Estate               2
Consumer Staples          1
Communication Services    1


,ticker,company,sector,cik,total_10k,earliest_year,latest_year,error
63,TECH,Bio-Techne,Health Care,0000842023,16,2010.0,2025.0,None
75,BRO,Brown & Brown,Financials,0000079282,16,2011.0,2026.0,None
82,CPT,Camden Property Trust,Real Estate,0000906345,16,2011.0,2026.0,None
89,CASY,Casey's,Consumer Staples,0000726958,15,2011.0,2025.0,None
108,CTAS,Cintas,Industrials,0000723254,16,2010.0,2025.0,None
128,CPRT,Copart,Industrials,0000900075,16,2010.0,2025.0,None
165,SATS,EchoStar,Communication Services,0001415404,16,2011.0,2026.0,None
195,FRT,Federal Realty Investment Trust,Real Estate,0000034903,17,2010.0,2026.0,None
246,IEX,IDEX Corporation,Industrials,0000832101,16,2011.0,2026.0,None
265,JKHY,Jack Henry & Associates,Financials,0000779152,16,2010.0,2025.0,None


## Step 5 — Select your final company list

Strategy: pick a **balanced cross-sector sample**.
- Enough companies per sector to capture diverse risk types
- Total: ~100 companies gives ~1,400 documents (100 × 14 years) — enough for BERTopic

Adjust `N_PER_SECTOR` based on how many well-covered companies you found above.

In [21]:
N_PER_SECTOR = 10  # companies per sector — adjust as needed

if well_covered.empty:
    print("No companies passed the filter — loosen MIN_FILINGS or earliest_year in Step 4.")
    final_selection = well_covered.copy()
else:
    final_selection = pd.concat(
        [g.sample(min(len(g), N_PER_SECTOR), random_state=42)
         for _, g in well_covered.groupby("sector")],
        ignore_index=True,
    )

    print(f"Final company selection: {len(final_selection)} companies")
    print(f"Estimated documents: ~{len(final_selection) * 14} (14 years × {len(final_selection)} companies)")
    print(f"\nBreakdown by sector:")
    print(final_selection["sector"].value_counts().to_string())

    # Save the selection for use in the main pipeline
    final_selection.to_csv("data/selected_companies.csv", index=False)
    print("\nSaved to data/selected_companies.csv")

    # display(final_selection[["ticker", "company", "sec
    final_selection[["ticker", "company", "sector", "cik", "total_10k", "earliest_year", "latest_year"]]

Final company selection: 14 companies
Estimated documents: ~196 (14 years × 14 companies)

Breakdown by sector:
sector
Consumer Discretionary    3
Industrials               3
Financials                2
Health Care               2
Real Estate               2
Communication Services    1
Consumer Staples          1

Saved to data/selected_companies.csv


## Step 6 — Spot-check one company in EDGAR

Before running the full pipeline, manually inspect one company's filing list
to confirm the data looks as expected.

In [22]:
def inspect_company(cik: str, company_name: str):
    """
    Print the 10-K filing history for one company.
    Useful for manually verifying data availability.
    """
    url = f"https://data.sec.gov/submissions/CIK{cik}.json"
    response = requests.get(url, headers=HEADERS)
    data = response.json()

    filings = data["filings"]["recent"]
    df = pd.DataFrame({
        "form":       filings["form"],
        "date":       filings["filingDate"],
        "accession":  filings["accessionNumber"],
        "document":   filings["primaryDocument"],
    })

    ten_k = df[df["form"] == "10-K"].copy()
    ten_k["year"] = pd.to_datetime(ten_k["date"]).dt.year
    ten_k = ten_k[ten_k["year"] >= 2010]

    print(f"\n{'='*50}")
    print(f"Company : {company_name}")
    print(f"CIK     : {cik}")
    print(f"10-K filings from 2010: {len(ten_k)}")
    print(f"{'='*50}")
    return ten_k[["year", "date", "accession", "document"]]


# Inspect the first company in your selection
first = final_selection.iloc[0]
inspect_company(first["cik"], first["company"])


Company : EchoStar
CIK     : 0001415404
10-K filings from 2010: 16


,year,date,accession,document
34,2026,2026-03-02,0001104659-26-021817,tmb-20251231x10k.htm
147,2025,2025-02-27,0001558370-25-001663,tmb-20241231x10k.htm
233,2024,2024-02-29,0001558370-24-002209,tmb-20231231x10k.htm
343,2023,2023-02-23,0001415404-23-000005,sats-20221231.htm
401,2022,2022-02-24,0001415404-22-000005,sats-20211231.htm
438,2021,2021-02-23,0001415404-21-000005,sats-20201231.htm
474,2020,2020-02-20,0001415404-20-000005,sats12311910kdocument1.htm
526,2019,2019-02-21,0001415404-19-000003,sats12311810kdocument.htm
567,2018,2018-02-22,0001415404-18-000005,sats12311710kdocument.htm
620,2017,2017-02-24,0001415404-17-000010,sats_123116x10kdocument.htm
